# Reasoning-knob probe for the LLM Mesh

**Question this notebook answers:** on *this* DSS instance, is there any way to turn up
reasoning depth (extended thinking / reasoning effort) for the two models we use -
Claude Sonnet 4.5 (Bedrock) and GPT 5.2 (Azure OpenAI)?

Background: the production code (`python/utilities/llm.py`, `review_table.py`) passes only
`temperature` / `timeout` / `max_tokens` - no reasoning setting exists anywhere in the repo.
Dataiku added reasoning controls in DSS 14.3 (reasoning budget on OpenAI/Azure/Bedrock/Vertex)
and DSS 14.4.2 (connection-level default reasoning effort), but the project logs show this
instance is DSS **13.5.4**, where none of that is documented. Docs do not say what 13.5 does
with unknown completion-settings keys (silently drop? reject?), so we measure instead of guess.

**What it does:** for each model it sends the same tiny puzzle prompt under ~8 different
completion-settings variants (baseline, production combo, and every plausible spelling of a
reasoning knob). For each call it records: accepted or error, latency, reply length, and any
token-usage fields the response exposes. A verdict cell then compares each variant against
the model's baseline:

- **errored** - the Mesh or provider rejected the setting (the error text says which side);
- **accepted, no effect** - same latency/usage as baseline, so the key was silently dropped;
- **accepted, effective** - materially higher latency and/or reasoning-token usage, i.e. a
  real knob you can copy into `COMPLETION_SETTINGS` of the induction notebook.

It also flags the reverse risk: if GPT 5.2 rejects `temperature`/`maxOutputTokens` (reasoning
models often do on chat-completions), the induction run must use `COMPLETION_SETTINGS = {}`.

**Cost:** ~16 short completions (a few cents). Results print below and are uploaded to the
managed folder under `reasoning_probe/`.


In [ ]:
# ------------------------------- CONFIG -------------------------------------
import dataiku

OUTPUT_FOLDER = '3UkrB0N9'   # managed folder for the results artifact ('' = skip upload)

client = dataiku.api_client()
project = client.get_default_project()

try:
    dss_version = client.get_instance_info().raw.get('dssVersion', 'unknown')
except Exception:
    dss_version = 'unknown (get_instance_info unavailable)'
print('DSS version:', dss_version)

_vars = project.get_variables().get('local', {})
MODELS = {}
sonnet = _vars.get('default_llm_model') or 'bedrock:AWS-Bedrock:us.anthropic.claude-sonnet-4-5-20250929-v1:0'
gpt52 = _vars.get('alt_llm_id') or 'azureopenai:Azure-OpenAi-NoCache:gpt-5.2'
MODELS['sonnet-4.5'] = sonnet
MODELS['gpt-5.2'] = gpt52
for name, mid in MODELS.items():
    print(f'{name}: {mid}')

# Short task with a known answer. Correctness is not the point (both models get it
# right at any effort) - the point is that reasoning, when actually enabled, shows
# up as extra latency and reasoning/thinking token usage on the SAME prompt.
PROMPT = ('A bat and a ball cost $1.10 in total. The bat costs $1.00 more than '
          'the ball. How much does the ball cost? Think it through, then reply '
          'with just the amount.')

# Every variant we can plausibly spell. Notes:
#  - reasoningEffort / reasoning_effort: effort-style knob (OpenAI-family naming)
#  - reasoningBudget: the name DSS 14.3 release notes use for the budget knob
#  - thinking{...}: Anthropic's raw extended-thinking parameter; requires
#    temperature=1 and budget < max output, so those ride along
VARIANTS = [
    ('baseline',              {}),
    ('prod_combo',            {'temperature': 0.2, 'maxOutputTokens': 4096}),
    ('temp_only',             {'temperature': 0.2}),
    ('max_out_only',          {'maxOutputTokens': 4096}),
    ('reasoningEffort_high',  {'reasoningEffort': 'high'}),
    ('reasoning_effort_high', {'reasoning_effort': 'high'}),
    ('reasoningBudget_2048',  {'reasoningBudget': 2048, 'maxOutputTokens': 4096}),
    ('thinking_2048',         {'thinking': {'type': 'enabled', 'budget_tokens': 2048},
                               'temperature': 1, 'maxOutputTokens': 4096}),
]


In [ ]:
# ------------------------------- HELPERS ------------------------------------
import json
import time

def _usage_fields(resp):
    """Best-effort: pull any token/usage info out of the response object.
    The raw payload shape differs across DSS versions, so scan generically for
    keys mentioning tokens/usage/reasoning/thinking rather than assuming one."""
    raw = None
    for attr in ('raw_resp', '_raw', 'raw'):
        raw = getattr(resp, attr, None)
        if isinstance(raw, dict):
            break
    if not isinstance(raw, dict):
        return {}
    found = {}
    def walk(obj, path):
        if isinstance(obj, dict):
            for k, v in obj.items():
                p = f'{path}.{k}' if path else k
                kl = k.lower()
                if isinstance(v, (int, float)) and ('token' in kl or 'usage' in kl
                                                    or 'reasoning' in kl or 'thinking' in kl):
                    found[p] = v
                walk(v, p)
        elif isinstance(obj, list):
            for i, v in enumerate(obj[:5]):
                walk(v, f'{path}[{i}]')
    walk(raw, '')
    return found

def probe_once(llm, settings, prompt):
    """One completion under one settings dict. An exception IS a result here
    (it tells us the key was rejected), so no retries and no re-raise."""
    row = {'ok': False, 's': None, 'reply_chars': None, 'reply_head': None,
           'usage': {}, 'error': None}
    t0 = time.monotonic()
    try:
        comp = llm.new_completion()
        try:
            comp.settings.update(settings)
        except Exception as e:  # client-side rejection (unlikely, but distinct)
            row['error'] = 'settings.update failed: ' + repr(e)[:200]
            return row
        comp.with_message(prompt)
        resp = comp.execute()
        row['s'] = round(time.monotonic() - t0, 2)
        text = (resp.text or '') if getattr(resp, 'success', False) else ''
        if not text:
            row['error'] = 'unsuccessful or empty completion'
            return row
        row.update(ok=True, reply_chars=len(text), reply_head=text[:80].replace('\n', ' '),
                   usage=_usage_fields(resp))
    except Exception as e:
        row['s'] = round(time.monotonic() - t0, 2)
        row['error'] = repr(e)[:300]
    return row


In [ ]:
# ------------------------------- PROBE RUN ----------------------------------
results = {}   # model -> variant -> row
for model_name, model_id in MODELS.items():
    print(f'=== {model_name}  ({model_id})')
    try:
        llm = project.get_llm(model_id)
    except Exception as e:
        print('  cannot get LLM handle:', repr(e)[:200])
        results[model_name] = {'_handle_error': repr(e)[:300]}
        continue
    results[model_name] = {}
    for variant, settings in VARIANTS:
        row = probe_once(llm, settings, PROMPT)
        row['settings'] = settings
        results[model_name][variant] = row
        status = 'ok' if row['ok'] else 'ERROR'
        extra = (f"{row['reply_chars']} chars, usage={row['usage']}" if row['ok']
                 else row['error'])
        print(f"  {variant:24s} {status:5s} {row['s'] if row['s'] is not None else '-':>6}s  {extra}")


In [ ]:
# ------------------------------- VERDICT ------------------------------------
# 'effective' = accepted AND (>=1.6x baseline latency OR reasoning/thinking
# tokens visible in usage). Latency on one call is noisy - treat a lone latency
# signal as 'probably', and rerun this cell if in doubt.
print(f'DSS version: {dss_version}')

def reasoning_tokens(usage):
    return sum(v for k, v in usage.items()
               if 'reasoning' in k.lower() or 'thinking' in k.lower())

recommendations = []
for model_name, rows in results.items():
    if '_handle_error' in rows:
        continue
    base = rows.get('baseline', {})
    base_s = base.get('s') or 0.001
    print(f'\n=== {model_name}')
    effective = []
    for variant, settings in VARIANTS:
        row = rows.get(variant)
        if row is None or variant == 'baseline':
            continue
        if not row['ok']:
            src = 'provider' if any(t in (row['error'] or '').lower()
                                    for t in ('400', 'invalid', 'unsupported')) else 'mesh/other'
            print(f'  {variant:24s} REJECTED ({src}): {row["error"][:120]}')
            continue
        rt = reasoning_tokens(row['usage'])
        ratio = (row['s'] or 0) / base_s
        if rt > 0 or ratio >= 1.6:
            effective.append(variant)
            why = f'{rt} reasoning tokens' if rt else f'{ratio:.1f}x baseline latency'
            print(f'  {variant:24s} accepted - LIKELY EFFECTIVE ({why})')
        else:
            print(f'  {variant:24s} accepted - no visible effect ({ratio:.1f}x latency), '
                  'key probably dropped')
    # what to do in the induction notebook
    prod = rows.get('prod_combo', {})
    if not prod.get('ok'):
        recommendations.append(f"{model_name}: production settings (temperature/maxOutputTokens) "
                               f"FAIL on this model - run induction with COMPLETION_SETTINGS = {{}}")
    for v in effective:
        recommendations.append(f"{model_name}: add {dict(VARIANTS)[v]} to COMPLETION_SETTINGS "
                               f"to raise reasoning depth")
    if not effective:
        recommendations.append(f"{model_name}: no working reasoning knob on this DSS - options: "
                               f"ask the admin to set it on the connection after upgrading "
                               f"(DSS 14.3+ budget, 14.4.2+ connection-level effort), or use "
                               f"prompt-side 'plan before answering' instructions")

print('\n--- Recommendations')
for r in recommendations:
    print(' *', r)


In [ ]:
# ------------------------------- ARTIFACT -----------------------------------
import io
import time as _t

if OUTPUT_FOLDER:
    artifact = {
        'generated_at': _t.strftime('%Y-%m-%dT%H:%M:%SZ', _t.gmtime()),
        'dss_version': dss_version,
        'models': MODELS,
        'prompt': PROMPT,
        'results': results,
        'recommendations': recommendations,
    }
    folder = dataiku.Folder(OUTPUT_FOLDER)
    path = f"reasoning_probe/{_t.strftime('%Y%m%d-%H%M%S', _t.gmtime())}_probe.json"
    with folder.get_writer(path) as w:
        w.write(json.dumps(artifact, indent=1, default=str).encode('utf-8'))
    print('uploaded', path)
else:
    print('OUTPUT_FOLDER empty - skipping upload')
